Fingerprinting

This workbook uses a IDL_python bridge to run fingerprinting analysis on seasonal runoff data across New Zealand. It defines functions which read previously created runoff datafiles and setup the data_obs, data_scen, and noise vectors. These vectors are then input into Daithi Stone's gendetec IDL code which returns the regression coefficients for the multiple linear regression problem as follows: data_obs = beta_est * data_scen + residual. 

Results are saved in ~/figures/fingerprinting/beta_est/ if using a single truncation and ""/scan_beta_est/ if depicting a range of truncation options. 

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from functools import reduce
import matplotlib.pyplot as plt
from idlpy import *
from scipy import stats
import statsmodels.api as sm


import seaborn as sns
import cartopy.crs as ccrs

ens_list = ['r1i1p1','r1i1p11','r1i1p13','r1i1p15','r1i1p3','r1i1p5','r1i1p7','r1i1p9',
            'r1i1p10','r1i1p12','r1i1p14','r1i1p2','r1i1p4','r1i1p6','r1i1p8']

IDL 8.7.2 (linux x86_64 m64).
(c) 2019, Harris Geospatial Solutions, Inc.

A new version is available: IDL 8.9
https://www.l3harrisgeospatial.com/Product-Downloads/
 
Licensed for use by: NIWA - National Institute of Water and Atmospheric Research Ltd - Ashley
License: 229483-BUF
Expiration date: 31-Mar-2024


In [2]:
def get_obs_data(season, mem, resample_interval):
    
    da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/lmr/tasmax/seasonal/HadGEM3_all_'+season+'_tasmax_LMR_'+mem+'.nc')

    if resample_interval != 'annual':
        da = da.resample(time=resample_interval).mean()

    da = da.stack(pixel=("lat", "lon"))
    da = da.dropna("pixel", how="all")        
    
    array = da.values.flatten()
    
    da.close()
    
    return(array)


In [8]:
get_obs_data('winter','r1i1p1','annual').shape

(6160,)

In [12]:
get_mean_and_noise('winter','all','r1i1p1','annual')[1].shape

(14, 6160)

In [3]:
def get_mean_and_noise(season,scen, mem, resample_interval):
    
    mean_for_noise = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/lmr/tasmax/seasonal/HadGEM3_'+scen+'_'+season+'_tasmax_LMR_ensMean.nc')

    if resample_interval != 'annual':
        mean_for_noise = mean_for_noise.resample(time=resample_interval).mean()
        
    noise = []
    da_list = []
    for mem_name in ens_list:

        if mem_name == mem:
            continue

        mem_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/lmr/tasmax/seasonal/HadGEM3_'+scen+'_'+season+'_tasmax_LMR_'+mem_name+'.nc')

        if resample_interval != 'annual':
            mem_da = mem_da.resample(time=resample_interval).mean()
            
        diff = mem_da - mean_for_noise

        diff = diff.stack(pixel=("lat", "lon"))# Create a MultiIndex
        diff = diff.dropna("pixel", how="all")# Drop the pixels that only have NA values.
        
        diff_array = diff.values.flatten()
        diff.close()
        
        mem_da.close()
        
        da_list.append(mem_da)
        noise.append(diff_array)

    mean_for_noise.close()
    
    ens_mean = sum(da_list)/len(da_list)
    ens_mean = ens_mean.stack(pixel=("lat", "lon")) # Create a MultiIndex
    ens_mean = ens_mean.dropna("pixel", how="all") # Drop the pixels that only have NA values.
    
    ens_mean_array = ens_mean.values.flatten()
    
    ens_mean.close()
    
    return((ens_mean_array, np.array(noise)))


In [18]:
mean, noise = get_mean_and_noise('summer','nat', 'r1i1p1', 'annual')

In [25]:
noise.shape

(14, 6160)

In [14]:
'''
Perfect model testing for EACH SEASON INDIVIDUALLY, using each ensemble member as the observations.
'''

seasons = ['summer','autumn','winter','spring']

for resample_interval in ['annual','5Y','10Y']:

    ensemble_member = []
    ant_upper = []
    ant_lower = []
    ant_best = []
    nat_upper = []
    nat_lower = []
    nat_best = []
    season = []
    for mem_name in ens_list:

        fig,axs = plt.subplots(2,2, figsize=(12,10))
        axs = axs.flatten()

        for i,s in enumerate(seasons):    

            IDL.retall

            all_ens_mean,noise_all = get_mean_and_noise(s,'all',mem_name,resample_interval)
            nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',mem_name,resample_interval)

            data_obs = get_obs_data(s, mem_name, resample_interval)

            data_scen = np.array([all_ens_mean,nat_ens_mean])

            # double noise estimate
            data_noise_1 = np.concatenate((noise_all,noise_nat))
            data_noise_2 = np.concatenate((noise_all,noise_nat))

            #data_noise_1 = np.concatenate((noise_all[:7],noise_nat[:7]))
            #data_noise_2 = np.concatenate((noise_all[7:],noise_nat[7:]))
            
            trunc = np.array([i for i in range(3,26)])

            IDL.data_obs = data_obs
            IDL.data_scen = data_scen
            IDL.data_noise_1 = data_noise_1
            IDL.data_noise_2 = data_noise_2
            IDL.trunc = trunc
            IDL.scan_beta_est = []
            IDL.scan_p_resid = []
            IDL.scan_resid_sumsq = []
            IDL.p_resid = []
            IDL.resid_sumsq = []
            IDL.noise_singval = []
            IDL.z_best = []

            IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est, scan_resid_sumsq=scan_resid_sumsq, \
                    data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc, scan_beta_est=scan_beta_est,\
                    scan_p_resid=scan_p_resid,noise_singval=noise_singval,z_best=z_best")

            beta_est = IDL.beta_est
            scan_beta_est = IDL.scan_beta_est
            scan_p_resid = IDL.scan_p_resid
            scan_resid_sumsq = IDL.scan_resid_sumsq
            noise_singval = IDL.noise_singval
            z_best = IDL.z_best

            for j in range(len(scan_beta_est)):

                beta_est = scan_beta_est[j]

                best_1,low_1,up_1 = beta_est[0][0],beta_est[1][0],beta_est[2][0]
                best_2,low_2,up_2 = beta_est[0][1],beta_est[1][1],beta_est[2][1]

                axs[i].scatter([j+3]*3,[low_2,best_2,up_2],c='blue',alpha=0.7,label='nat' if j == 0 else '')
                axs[i].scatter([j+3]*3,[low_1,best_1,up_1],c='red',label='ant' if j == 0 else '')

                axs[i].vlines(j+3,low_2,up_2,colors=['blue'])
                axs[i].vlines(j+3,low_1,up_1,colors=['red'])
                
                axs[i].axhline(0,-0.5,1.5)
                axs[i].axhline(1,-0.5,1.5)


            t = 10
            ensemble_member.append(mem_name)
            ant_upper.append(scan_beta_est[t-3][2][0])
            ant_lower.append(scan_beta_est[t-3][1][0])
            ant_best.append(scan_beta_est[t-3][0][0])
            nat_upper.append(scan_beta_est[t-3][2][1])
            nat_lower.append(scan_beta_est[t-3][1][1])
            nat_best.append(scan_beta_est[t-3][0][1])
            season.append(s)
            
            axs[i].set_title(s)

            axs[i].legend()
            axs[i].set_ylim(-1.2,2.5)
        

        plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/HadGEM3/tasmax/seasonal/'+resample_interval+'/' + 'PM_' + mem_name + '.png')

        plt.close(fig)
        

    df = pd.DataFrame({'ensemble_member':ensemble_member,
                      'season':season,
                      'ant_upper':ant_upper,
                      'ant_lower':ant_lower,
                      'ant_best':ant_best,
                      'nat_upper':nat_upper,
                      'nat_lower':nat_lower,
                      'nat_best':nat_best})

    for sort_type in ['nat_best','ant_best']:
        
        fig,axs = plt.subplots(2,2, figsize=(12,10))
        axs = axs.flatten()
        
        for i,s in enumerate(seasons):
            
            sorted_df = df.loc[df['season'] == s].sort_values(by=sort_type)
            
            axs[i].scatter([i for i in range(len(sorted_df.nat_upper))],sorted_df.nat_upper,c='blue',label='nat')
            axs[i].scatter([i for i in range(len(sorted_df.nat_lower))],sorted_df.nat_lower,c='blue',label='')
            axs[i].scatter([i for i in range(len(sorted_df.nat_best))],sorted_df.nat_best,c='blue',label='')

            axs[i].scatter([i for i in range(len(sorted_df.ant_upper))],sorted_df.ant_upper,c='red',label='ant')
            axs[i].scatter([i for i in range(len(sorted_df.ant_lower))],sorted_df.ant_lower,c='red',label='')
            axs[i].scatter([i for i in range(len(sorted_df.ant_best))],sorted_df.ant_best,c='red',label='')

            axs[i].vlines([i for i in range(len(sorted_df.nat_lower))],sorted_df.nat_lower,sorted_df.nat_upper,colors=['blue'],label='')
            axs[i].vlines([i for i in range(len(sorted_df.ant_lower))],sorted_df.ant_lower,sorted_df.ant_upper,colors=['red'],label='')

            del sorted_df
            
            axs[i].axhline(0,-0.5,1.5)
            axs[i].axhline(1,-0.5,1.5)

            axs[i].set_title(s + ', trunc = ' + str(t))

            axs[i].legend()
            
            axs[i].set_ylim(-1,3)

        plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/HadGEM3/tasmax/seasonal/' + resample_interval + '/' + 'PM_ensemble_summary_'+sort_type+'_sort.png')

        plt.close(fig)
        
    df.to_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/HadGEM3/summary_csv/' + 'tasmax_'+resample_interval+'_truncation_' + str(t)+'.csv')


In [14]:
all_ens_mean,noise_all = get_mean_and_noise(s,'all',mem_name,resample_interval)
nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',mem_name,resample_interval)

data_obs = get_obs_data(s, mem_name, resample_interval)

data_scen = np.array([all_ens_mean,nat_ens_mean])

data_noise_1 = np.concatenate((noise_all[:7],noise_nat[:7]))
data_noise_2 = np.concatenate((noise_all[7:],noise_nat[7:]))


trunc = np.array([i for i in range(3,13)])

10

In [6]:
print(all_ens_mean.shape)
print(nat_ens_mean.shape)

print(noise_all.shape)
print(noise_nat.shape)

print(data_noise_1.shape)
print(data_noise_2.shape)

(784,)
(784,)
(14, 784)
(14, 784)
(14, 784)
(14, 784)


In [5]:
print(trunc)
print(IDL.trunc)
print(len(IDL.scan_beta_est))

print(scan_beta_est == IDL.scan_beta_est)


[ 3  4  5  6  7  8  9 10 11 12]
12
10
[[[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]

 [[ True  True]
  [ True  True]
  [ True  True]]]


In [6]:
'''
Ensemble summary at one truncation level
'''

seasons = ['summer','autumn','winter','spring']

for resample_interval in ['annual','5Y','10Y']:

    season_dfs = []
    
    for s in seasons:

        ensemble_members = []
        ant_uppers = []
        ant_lowers = []
        ant_bests = []

        nat_uppers = []
        nat_lowers = []
        nat_bests = []
        for mem_name in ens_list:

            ensemble_members.append(mem_name)

            IDL.retall

            #print('before get_mean all')
            all_ens_mean,noise_all = get_mean_and_noise(s,'all',mem_name,resample_interval)
            #print('before get_mean nat')
            nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',mem_name,resample_interval)
   
            #print('before get_obs')
            data_obs = get_obs_data(s, mem_name, resample_interval)

            data_scen = np.array([all_ens_mean,nat_ens_mean])

            data_noise_1 = np.concatenate((noise_all[:7],noise_nat[:7]))
            data_noise_2 = np.concatenate((noise_all[7:],noise_nat[7:]))

            trunc = 9

            IDL.data_obs = data_obs
            IDL.data_scen = data_scen
            IDL.data_noise_1 = data_noise_1
            IDL.data_noise_2 = data_noise_2
            IDL.trunc = trunc

            #print('before IDL run')
            IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est,\
                    data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc")

            beta_est = IDL.beta_est

            best_ant,low_ant,up_ant = beta_est[0][0],beta_est[1][0],beta_est[2][0]
            best_nat,low_nat,up_nat = beta_est[0][1],beta_est[1][1],beta_est[2][1]

            ant_uppers.append(up_ant)
            ant_lowers.append(low_ant)
            ant_bests.append(best_ant)

            nat_uppers.append(up_nat)
            nat_lowers.append(low_nat)
            nat_bests.append(best_nat)        

        df = pd.DataFrame({'ensemble_member':ensemble_members,
                          'ant_uppers':ant_uppers,
                          'ant_lowers':ant_lowers,
                          'ant_bests':ant_bests,
                          'nat_uppers':nat_uppers,
                          'nat_lowers':nat_lowers,
                          'nat_bests':nat_bests})

        season_dfs.append(df)
        
    for sort_type in ['nat_bests','ant_bests']:
        
        fig,axs = plt.subplots(2,2, figsize=(12,10))
        axs = axs.flatten()
        
        for i,s in enumerate(seasons):
            
            sorted_df = season_dfs[i].sort_values(by=sort_type)
            
            axs[i].scatter([i for i in range(len(sorted_df.nat_uppers))],sorted_df.nat_uppers,c='blue',label='nat')
            axs[i].scatter([i for i in range(len(sorted_df.nat_lowers))],sorted_df.nat_lowers,c='blue',label='')
            axs[i].scatter([i for i in range(len(sorted_df.nat_bests))],sorted_df.nat_bests,c='blue',label='')

            axs[i].scatter([i for i in range(len(sorted_df.ant_uppers))],sorted_df.ant_uppers,c='red',label='ant')
            axs[i].scatter([i for i in range(len(sorted_df.ant_lowers))],sorted_df.ant_lowers,c='red',label='')
            axs[i].scatter([i for i in range(len(sorted_df.ant_bests))],sorted_df.ant_bests,c='red',label='')

            axs[i].vlines([i for i in range(len(sorted_df.nat_lowers))],sorted_df.nat_lowers,sorted_df.nat_uppers,colors=['blue'],label='')
            axs[i].vlines([i for i in range(len(sorted_df.ant_lowers))],sorted_df.ant_lowers,sorted_df.ant_uppers,colors=['red'],label='')

            del sorted_df
            
            axs[i].axhline(0,-0.5,1.5)
            axs[i].axhline(1,-0.5,1.5)

            axs[i].set_title(s + ', trunc = ' + str(trunc))

            axs[i].legend()

        plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/HadGEM3/tasmax/seasonal/' + resample_interval + '/' + 'PM_ensemble_summary_'+sort_type+'_sort.png')

        plt.close(fig)
        
    del season_dfs


In [8]:
season_dfs

[   ensemble_member  ant_uppers  ant_lowers  ant_bests  nat_uppers  nat_lowers  \
 0               01    2.945503   -1.171685   0.886909    1.418537   -0.913303   
 1               02    2.141757   -2.055774   0.042991    2.095440   -0.389146   
 2               03    0.964600   -3.142311  -1.088855    0.845112   -1.642260   
 3               04    3.776680   -0.454204   1.661238    1.727275   -0.674883   
 4               05    3.084670   -1.127310   0.978680    1.999489   -0.319021   
 5               06    3.729710   -0.559522   1.585094    2.912770    0.403448   
 6               07    5.282249    0.836452   3.059351    2.495355    0.103870   
 7               08    2.617718   -1.511528   0.553095    2.645419    0.263108   
 8               09    1.422794   -2.466259  -0.521732    0.757684   -1.347344   
 9               10    2.441251   -1.706975   0.367138    1.255121   -1.126106   
 10              11   -0.202512   -4.046957  -2.124735    2.536064    0.176041   
 11             